# Earth's elevation has two peaks. So does Mars's. Same reason?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A//github.com/AI4EPS/EPS88_PyEarth&branch=main&urlpath=lab/tree/EPS88_PyEarth/docs/notebooks/03_two_peaks.ipynb).

Ask how high every point on a planet's surface is, and you might expect the answers to crowd around one typical height, the way people's heights do. Earth's do not: they pile up at two levels, with a trough between them that little of the planet sits in. Mars, which has no ocean and no moving plates, has two levels as well.

Today both planets arrive as grids of numbers, one elevation for every one-degree square of the surface. You will find where each planet's two levels sit, map them, measure how much of Earth lies below sea level, and then open a table of earthquakes to see where Earth's old ocean floor goes.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by a cell that already has the shape of the answer written in, with `...` wherever your
code goes. Replace every `...`, run every cell so your answers and figures are
saved in the file, then **download this notebook itself — the `.ipynb` file — and upload it
to Gradescope.** Not a PDF: the marking reads your notebook, and a PDF cannot be read.
In JupyterLab: **File ▸ Download**, or right-click the file in the left-hand panel and choose
**Download**.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## What you'll be able to do

**The science.** Find the two levels of Earth's and Mars's surfaces, see what makes Earth's, and
say what is known and what is still argued about Mars's.

**The code.** Arrays: `.shape` and `.size` · arithmetic on a whole grid at once · `.ravel()` ·
`np.histogram` and `.argmax()` · masks such as `earth < 0`, and `earth[mask]` to keep the cells
where one is True · `plt.imshow` to draw a grid as a picture. Tables: `.info()` · NaN ·
`.median()` · picking rows with a mask · `.groupby()`.

**Eight places where you write something: five in class, three at home.** Each is headed
*Your turn*, and the cell under it already has the shape of the answer written in, with `...`
wherever your code goes. Replace every `...`; a comment beside each one says what it is for. A
`...` left in place is not code, and the cell will not do what you expect until it is replaced.

1. Where do Earth's two levels sit?
2. How much of Earth lies below sea level?
3. Where are Mars's two levels?
4. Where does old ocean floor go?

## Setup

Run the next cell once; you are not expected to follow all of it. It loads `earth` and `mars`,
one elevation in metres for every one-degree square of each planet, the coastlines for maps, and
`quakes`, the USGS catalogue of every earthquake of magnitude 5.5 or more from 2000 to 2025. It
also makes `bins`, the edges of 250-metre histogram bins, and two grids that section 2 explains,
`latitude` and `cell_area` (the second uses `np.cos`, a cosine).

The counts written into the text below were taken from the catalogue when this notebook was
written. Your run asks USGS for the catalogue as it stands today, so a count may come out one or
two different; the self-checks allow for that.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

def load():
    """the USGS catalogue of magnitude 5.5 and larger earthquakes, 2000 to 2025: live if the network is up, the copy stored with the course if not"""
    # Ask the live archive first. If it is down, or you are offline, read the copy stored with
    # the course instead, so the notebook still runs.
    try:
        return pd.read_csv("https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv&orderby=time-asc"
                       "&starttime=2000-01-01&endtime=2026-01-01&minmagnitude=5.5")
    except Exception as e:
        print("live source unreachable, using the cached copy:", type(e).__name__)
        return pd.read_csv(CACHE + "/" + "week03_2000-01-01_2026-01-01_M5.5.csv")

# The two elevation grids ship with the course, so they are read straight from it. One row per
# degree of latitude (row 0 is the far north), one column per degree of longitude (column 0 is
# 180 degrees west). Earth's come from NOAA's ETOPO relief model, Mars's from the MOLA laser
# altimeter on Mars Global Surveyor.
earth = pd.read_csv(CACHE + "/earth_elevation.csv", header=None).values
mars = pd.read_csv(CACHE + "/mars_elevation.csv", header=None).values
coast = pd.read_csv(CACHE + "/coastlines.csv")
quakes = load()

bins = np.arange(-10000, 21000, 250)                  # histogram bin edges, 250 m apart
# The latitude of every cell's centre, and each cell's area compared with a cell on the equator.
latitude = np.repeat(np.arange(89.5, -90, -1), 360).reshape(180, 360)
cell_area = np.cos(np.deg2rad(latitude))

print("elevation grids:", earth.shape, mars.shape, "  earthquakes:", len(quakes))

## 1. Where do Earth's two levels sit?

`earth` holds one elevation, in metres, for every one-degree square of Earth's surface: 180 rows
of latitude, the far north in row 0, and 360 columns of longitude. That is 64,800
numbers.

A list could hold all of them. Watch what a list does with arithmetic.

In [ ]:
heights_list = [0, 1000, 2000]
heights_array = np.array([0, 1000, 2000])

print("list  times 2:", heights_list * 2)
print("array times 2:", heights_array * 2)

The list did not double anything: it made a longer list with the same numbers twice. The array
doubled every one.

**Array.** A grid of numbers where every cell is the same kind of thing, so one line of arithmetic changes all of them at once.

So `earth / 1000` would turn every one of Earth's elevations into kilometres in one line. The
array will also say how it is laid out.

In [ ]:
print("shape:", earth.shape)   # rows, then columns
print("size: ", earth.size)    # how many numbers that is altogether

`earth.shape` is the layout, rows first: 180 bands of latitude, each one 360 cells of longitude.
`earth.size` is the same numbers counted straight through, all 64,800 of them.

One quantity, measured everywhere on a regular mesh — that is what makes elevation a grid.
Nothing in `earth` stores a latitude or a longitude: the position in the grid *is* the position
on the planet. Row 0 is the far north, column 0 is 180 degrees west, one degree of each per cell.

Which is why it can be drawn as a picture with no coordinates handed to it. `plt.imshow` puts one
pixel per cell, `extent` says what longitudes and latitudes the edges are, and `plt.colorbar`
adds the key that says what the colours mean.

In [ ]:
plt.imshow(earth, extent=[-180, 180, -90, 90], vmin=-6000, vmax=6000)   # colours stop at ±6000 m
plt.colorbar(label="elevation (m)")
plt.xlabel("longitude (degrees east)")
plt.ylabel("latitude (degrees north)")
plt.title(f"Earth ({earth.size:,} cells)")
plt.show()

Ocean floor over most of the planet, at what looks like one fairly even depth, with the
continents standing above it and a few bright places higher still.

**Before you run the next cell, decide what you expect.** If you counted how many of the
64,800 cells sit at each height, what shape would the counts make — one hump around a
typical height, the way people's heights do, or something else?

Week 1 counted how many earthquakes fell at each magnitude, and this is the same count asked of
elevations. A histogram wants one long line of numbers, not a grid: `.ravel()` lays the grid out
flat, row after row. `bins`, from setup, holds bin edges 250 metres apart, the same for both
planets, so a level means the same thing each time it is quoted.

In [ ]:
plt.hist(earth.ravel(), bins=bins)
plt.xlim(-9500, 6500)                  # the bins run on to 21,000 m, for Mars
plt.xlabel("elevation (m)")
plt.ylabel("number of one-degree cells")
plt.title(f"Earth, {earth.size:,} cells, 250 m bins")
plt.show()

Two humps with a trough between them, one a few kilometres below sea level and one close to it.
A third, much smaller bump sits near 3,000 m — the Antarctic ice sheet. Section 2 comes back to
why it looks bigger here than it should.

What a histogram shows depends on its bins, though. The next cell draws the same grid in only
four bins and counts them too. `np.histogram` does the counting `plt.hist` does and hands back
two arrays: the counts, and the edges the bins run between.

Four bins need five edges, so `edges` is one longer than `counts`. `edges[0]` is where bin 0
starts, `edges[1]` where bin 1 starts, and so on — a number in square brackets picks out one
value by its position. `.argmax()` gives the *position* of the largest value, the way
`list.index(max(list))` does for a list, so that position in the brackets — `edges[counts.argmax()]`
— is where the tallest bin starts.

In [ ]:
plt.hist(earth.ravel(), bins=4)
plt.xlabel("elevation (m)")
plt.ylabel("number of one-degree cells")
plt.title(f"Earth, {earth.size:,} cells, 4 bins")
plt.show()

counts, edges = np.histogram(earth.ravel(), bins=4)
print("cells in each bin:", counts)
print("bin edges, in m:  ", edges)
print("the tallest is bin", counts.argmax())
print("which starts at   ", edges[counts.argmax()], "m")

### ✏️ Your turn 1

Four bins give one hump: the counts climb to one tallest bin and fall away, with no dip between
two humps. Is that the planet, or the bins? The cell below counts the same grid in five bins and
prints the edges in kilometres, one line of arithmetic for all six. Then it asks whether the
middle bin is a dip, with a small function: `is_dip(left, middle, right)` is True when the middle
count is lower than the count on its left **and** lower than the count on its right. Fill in the
two comparisons.

So: do five bins show two humps where four showed one?

**Use these names**, because the self-check looks for them: `is_dip`.

In [ ]:
# ← your answer here: replace every ... with code
counts, edges = np.histogram(earth.ravel(), bins=5)
print("cells in each bin:", counts)
print("bin edges, in km: ", edges / 1000)


def is_dip(left, middle, right):
    """True when the middle count is lower than the counts on both sides of it"""
    return ... and ...              # lower than the count on its left, and lower than the one on its right


dip = is_dip(counts[1], counts[2], counts[3])
print("five bins show a dip between two humps:", dip)

In [ ]:
assert is_dip(2, 1, 3) == True, "is_dip(2, 1, 3) should be True: 1 is lower than both 2 and 3"
assert is_dip(3, 1, 2) == True, "is_dip(3, 1, 2) should be True: 1 is lower than both 3 and 2 — compare middle with each side"
assert is_dip(1, 2, 3) == False, \
    "is_dip(1, 2, 3) should be False: 2 is lower than the 3 on its right but not the 1 on its left, so both tests are needed"
assert is_dip(3, 2, 1) == False, \
    "is_dip(3, 2, 1) should be False: 2 is lower than the 3 on its left but not the 1 on its right, so both tests are needed"
assert dip == True, "dip should be True for the five real counts — is_dip must compare the middle count with each side"
print(f"✓ five bins — the middle bin holds {counts[2]} cells, fewer than the {counts[1]} and {counts[3]} beside it: a dip")

250-metre bins show two clear humps, so the next step is to find where each one peaks — which
means picking out the cells below the trough and the cells above it, separately.

**Mask.** Comparing an array with a number asks the same question of every cell at once and hands back a grid of True and False.

In [ ]:
high = earth > 5000                        # one True or False for every cell
print("the mask:                ", high.shape)
print("the elevations it keeps:", earth[high].shape)
print("the first five of them: ", earth[high][:5])
print("how many of those are north of the equator:", (high & (latitude > 0)).sum())

The mask has the same 180 by 360 layout as `earth`. Put it inside square brackets and you get the
elevations themselves, as one long line of numbers — the grid's shape is gone, because the cells
it kept are scattered all over it.

The last line asks two questions of every cell at once. `&` joins two masks cell by cell, so
`a & b` is True only where both are, and `.sum()` counts the Trues because Python counts True
as 1.

### ✏️ Your turn 2

-1000 m sits in the trough between the two humps, so the deep level is the tallest 250-metre bin
among the cells below -1000 m, and the high level the tallest among the rest.

The cell below picks out the two sets of cells, writes the counting once as the function
`tallest_bin(values, bin_edges)` — everything it needs comes in through its brackets, the way
`check_planet` did in week 2: the elevations themselves rather than a mask, and the edges to
count them into. The middle of a bin is its left edge plus half the 250-metre width. Fill in the
two selections, what to count, and which edge to return.

So: where do Earth's two levels sit?

**Use these names**, because the self-check looks for them: `deep_cells`, `high_cells`,
`earth_deep` and `earth_high`.

In [ ]:
# ← your answer here: replace every ... with code
deep_cells = earth[...]                         # the elevations of the cells below -1000 m
high_cells = earth[...]                         # the elevations of the cells at -1000 m or above


def tallest_bin(values, bin_edges):
    """the middle of the tallest 250-metre bin among these elevations"""
    counts, edges = np.histogram(..., bins=bin_edges)   # count these elevations, into these bins
    return edges[...] + 125                             # the left edge of the bin that holds the most, plus half a bin


earth_deep = tallest_bin(deep_cells, bins)
earth_high = tallest_bin(high_cells, bins)
print("Earth's deep level:", earth_deep, "m")
print("Earth's high level:", earth_high, "m")

In [ ]:
assert deep_cells.max() < -1000 <= high_cells.min(), \
    "deep_cells should hold only elevations below -1000 m, and high_cells only those at -1000 m or above"
assert deep_cells.size + high_cells.size == earth.size, \
    "between them, deep_cells and high_cells should hold every cell of earth: all of it below -1000 m, and all the rest"
assert np.size(earth_deep) == 1 and np.size(earth_high) == 1, \
    "earth_deep and earth_high should each be one number: return one edge, not all of them"
assert -4500 < earth_deep < -4250, \
    "earth_deep should be an elevation in metres: the middle of the tallest bin among deep_cells"
assert 0 < earth_high < 250, \
    "earth_high should be the middle of the tallest bin among high_cells"
print(f"✓ Earth's two levels — {earth_deep} m and {earth_high} m, {earth_high - earth_deep} m apart")

## 2. How much of Earth lies below sea level?

`earth < 0` is a mask like the ones in section 1, and because it holds one True or False for every
one-degree square it can be drawn exactly as the elevations were — a grid is a grid. The coastline
goes on top, as on every map of Earth.

In [ ]:
below = earth < 0

plt.imshow(below, extent=[-180, 180, -90, 90], cmap="Greys")    # below sea level is dark
plt.plot(coast.lon, coast.lat, color="firebrick", lw=0.6)
plt.xlabel("longitude (degrees east)")
plt.ylabel("latitude (degrees north)")
plt.title(f"Earth below sea level, in dark ({below.size:,} cells)")
plt.show()

The dark region is one comparison, `earth < 0`, and it comes out as the oceans, with the
coastline along its edge. The map shows where the two levels are, ocean floor and continents;
what makes them is the crust beneath.

**Two kinds of crust.** Ocean crust is thin and dense, continental crust thick and light; both float on the mantle, and the thick, light kind floats higher.

Ocean crust is made at mid-ocean ridges, undersea mountain chains where two plates pull apart,
and sinks back into the mantle at subduction zones, where one plate dives under another.
Continental crust is too buoyant to go down that way, so most of it stays.

Adding up a mask counts its Trues, because Python counts True as 1, so
`below.sum() / below.size` is the share of the cells that lie below sea level.

### Predict before you run

What share of Earth's surface lies below sea level? Write your guess into `my_guess` as a
fraction, such as 0.5 for half, then run the next three cells.

In [ ]:
my_guess = None    # ← your number, written down before you look

In [ ]:
assert my_guess is not None, \
    "write a number into my_guess in the cell above — the commitment is the point, "\
    "and a guess you made before you saw the answer is the only one that can teach you anything"
print("✓ committed — I think", my_guess, "of Earth's surface lies below sea level")

In [ ]:
share_by_cells = below.sum() / below.size

print("you guessed:", my_guess)
print(f"share of the cells below sea level: {share_by_cells:.3f}")

If you guessed near 0.7, your guess was better than the count. Earth is usually quoted as about
seven-tenths ocean — not quite the same quantity as the share below sea level, but close — and
the grid says 0.660. That gap is not rounding.

Every cell counted as one, and the cells are not the same size. `cell_area`, made in setup, holds
each cell's area next to a cell on the equator, and it is a second grid laid over the first: same
180 by 360, and any row and column is the same square of ground in both. So a mask built from
`earth`, or from `latitude`, picks cells out of `cell_area` just as well.

In [ ]:
for degrees in [0, 30, 60, 89]:
    print("a cell at", degrees, "degrees covers", round(np.cos(np.deg2rad(degrees)), 2),
          "of what a cell on the equator covers")

south5 = latitude < -85                  # the five rows of cells nearest the South Pole
print()
print("those five rows are", round(south5.sum() / south5.size, 3), "of the cells")
print("but they cover     ", round(cell_area[south5].sum() / cell_area.sum(), 4), "of the surface")

A cell at 89 degrees covers a fiftieth of what a cell on the equator does, and counting cells
treated the two as equals. Those five southernmost rows are 0.028 of the
cells and 0.0019 of the surface — over-counted about fifteenfold.

That is what stretched Antarctica across the bottom of the map above, and what put the 3,000 m
bump in section 1's histogram — over half of that bump is those same five rows.

### ✏️ Your turn 3

**Area weighting.** A longitude-latitude grid counts every square once, but a square near the pole is a sliver; weight each row by cos(latitude) before quoting a percentage.

`cell_area[earth < 0]` is the areas of the cells below sea level, and `.sum()` adds them up.

The cell below writes the measurement once as `area_share(mask, areas)`. Both things it needs come
in through its brackets — the mask to apply and the grid of areas to weigh by — so the same
function works on either planet, and you can see what it is weighing. Fill in the two sides of
the division.

So: counted by area, how much of Earth lies below sea level?

**Use these names**, because the self-check looks for them: `area_share`.

In [ ]:
# ← your answer here: replace every ... with code
def area_share(mask, areas):
    """the share of a planet's surface where the mask is True, counting each cell by its area"""
    return ... / ...                 # the area where the mask is True, over the area of every cell


print("share of Earth's area below sea level:", round(area_share(earth < 0, cell_area), 3))

In [ ]:
assert 0.70 < area_share(earth < 0, cell_area) < 0.72, \
    "area_share(earth < 0, cell_area) should be the area of the cells below sea level over the area of all the cells"
assert 0.50 < area_share(mars < 0, cell_area) < 0.53, \
    "area_share must measure the mask it is given — Mars below its zero should not give Earth's share"
print(f"✓ area weighting — {area_share(earth < 0, cell_area):.3f} of Earth's surface is below sea level, not {share_by_cells:.3f}")

Counting by area moved the answer up because the rows near both poles, which the grid over-counts,
hold more land than the rows between: Antarctica across the bottom, and the far north of Canada,
Greenland and Siberia across the top. Over-counting land is under-counting ocean, and weighting
each cell by the ground it covers puts it back.

## 3. Where are Mars's two levels?

`mars` is the same kind of grid, 180 by 360. Mars has no sea, so its zero cannot be a sea level.
It is the *areoid*: the surface an ocean would settle onto if Mars had one, worked out from Mars's
gravity field. Earth's sea level is that same surface, with the water there to show where it is.

Mars's zero is not where its two levels divide, though. The trough between them sits about a
kilometre lower, which is why Your turn 4 splits Mars at -1000 m rather than at 0 — the same
line class used for Earth.

Instead of a below-and-above mask, the map below colours the whole range, and `plt.colorbar` adds
the key.

In [ ]:
plt.imshow(mars, extent=[-180, 180, -90, 90], vmin=-4000, vmax=4000)   # colours stop at ±4000 m
plt.colorbar(label="elevation (m)")
plt.xlabel("longitude (degrees east)")
plt.ylabel("latitude (degrees north)")
plt.title(f"Mars ({mars.size:,} cells)")
plt.show()

**Crustal dichotomy.** Mars's northern half sits kilometres lower than its southern half, and what made the step is still argued about.

The leading ideas are one or more giant impacts, or slow churning inside the young planet. The
next cell puts both planets on one histogram, with the same 250-metre bins.

In [ ]:
plt.hist(earth.ravel(), bins=bins, label="Earth")
plt.hist(mars.ravel(), bins=bins, label="Mars", alpha=0.6)      # see-through, so Earth shows
plt.xlim(-9500, 6500)                                            # Mars's highest cells run off the right
plt.xlabel("elevation (m)")
plt.ylabel("number of one-degree cells")
plt.title(f"Earth and Mars, {earth.size:,} cells each, 250 m bins")
plt.legend()
plt.show()

### ✏️ Your turn 4

Mars has two humps too, with a trough that also takes in -1000 m. The cell below picks out
Mars's two sets of cells, split at -1000 m as you did for Earth, finds each one's level with your
`tallest_bin`, handing it the same `bins`, and prints both planets' levels and the step between each pair. Fill in the two
selections.

So: are Mars's two levels further apart than Earth's?

**Use these names**, because the self-check looks for them: `mars_deep_cells`, `mars_high_cells`,
`mars_deep` and `mars_high`.

In [ ]:
# ← your answer here: replace every ... with code
mars_deep_cells = mars[...]                     # the elevations of Mars's cells below -1000 m
mars_high_cells = mars[...]                     # the elevations of Mars's cells at -1000 m or above
mars_deep = tallest_bin(mars_deep_cells, bins)
mars_high = tallest_bin(mars_high_cells, bins)

print("Mars's two levels: ", mars_deep, "m and", mars_high, "m, a step of", mars_high - mars_deep, "m")
print("Earth's two levels:", earth_deep, "m and", earth_high, "m, a step of", earth_high - earth_deep, "m")

In [ ]:
assert mars_deep_cells.max() < -1000 <= mars_high_cells.min(), \
    "mars_deep_cells should hold only elevations below -1000 m, and mars_high_cells only those at -1000 m or above"
assert mars_deep_cells.size + mars_high_cells.size == mars.size, \
    "between them, mars_deep_cells and mars_high_cells should hold every cell of mars: all of it below -1000 m, and all the rest"
assert mars_deep_cells.min() > -8000 and mars_high_cells.max() > 10000, \
    "the cells should be Mars's own: pick them out of mars, with a mask made from mars"
assert -4500 < mars_deep < -4250, \
    "mars_deep should be the tallest bin among mars_deep_cells"
assert 1250 < mars_high < 1500, \
    "mars_high should be the tallest bin among Mars's own cells at -1000 m or above"
print(f"✓ Mars's two levels — {mars_deep} m and {mars_high} m, {mars_high - mars_deep} m apart")

## 4. Where does old ocean floor go?

An elevation grid says where the ocean floor is. It cannot say where it goes. Ocean crust sinks
back into the mantle at subduction zones, and once it is under the surface there is no height
left to measure.

An earthquake catalogue can follow it down, because it records how deep each earthquake started.

A catalogue is not a grid, for two reasons. Earthquakes happen where and when they happen, not
on a regular mesh, so position in a list means nothing. And every row carries a time, a place, a
depth and a magnitude — different kinds of thing, where a grid holds one kind.

Swap two rows of the catalogue and nothing is lost, because each row carries its own latitude.
Swap two rows of the grid and you have moved a continent.

**Table.** A table with a name on every column, so you ask for data by name instead of by position.

You have been reading tables since week 1, and `quakes` is one. Both containers report their
`.shape`, and the two numbers mean different things.

What is new today is what to do with a table you have not seen before. `.info()` is the first
thing to run: every column, its type, and how many rows are not blank. `.median()` on a column
gives its middle value.

In [ ]:
print("earth, a grid: ", earth.shape, "-", earth.size, "numbers, every one an elevation in metres")
print("quakes, a table:", quakes.shape, "-", len(quakes), "earthquakes, each with",
      quakes.shape[1], "different kinds of fact")
print()
quakes.info()
print("rows with no dmin:", quakes["dmin"].isna().sum())
print("rows left if every row with a blank is dropped:", len(quakes.dropna()))
print("median depth (km):", quakes["depth"].median())

`depth` and `place` have no blanks, but `dmin` — how far away the nearest seismometer was — is
blank in 7,219 rows.

**NaN.** Where the file had nothing at all, pandas puts NaN. A NaN is a hole, not a zero.

`.isna()` is a mask marking the holes, and `.dropna()` throws away every row with a hole anywhere,
which here would leave 1,673 of 12,849 earthquakes — so ask for the
columns you need by name instead.

Most of these earthquakes are shallow: half are within 21 km of the surface,
and four in five within 70 km. The deep ones are the exception, and they are what this section is
about.

`place` names somewhere near each earthquake, with the country or region after the last comma.
`.str.split(", ")` cuts every place at its commas and `.str[-1]` keeps the last piece; assigning
that to a name the table does not have yet makes a new column. `.head()` shows a table's first
five rows, enough to check that it worked, and `.value_counts()` counts how often each value in a
column appears.

In [ ]:
quakes["region"] = quakes["place"].str.split(", ").str[-1]
print(quakes[["place", "region"]].head())
print(quakes["type"].value_counts())

### ✏️ Your turn 5

Nearly every row is an earthquake. Almost all earthquakes more than 500 km down happen inside
ocean plates sinking into the mantle, so they mark some of the places old ocean floor goes.

Picking rows works like picking cells: `quakes["mag"] >= 8` is a mask with one True or False per
row, and `quakes[quakes["mag"] >= 8]` keeps only the great earthquakes. `.groupby(column)` splits
a table into one group per value of that column, and `["depth"].count()` then counts the rows in
each group. `.sort_values()` puts a column or a set of counts in order, smallest first unless you
ask for `ascending=False`. The cell below keeps the earthquakes deeper than 500 km, counts them
region by region, and prints the six regions with the most. Fill in the mask and the column to
group by.

So: where are the earthquakes deeper than 500 km?

**Use these names**, because the self-check looks for them: `deep_quakes` and `per_region`.

In [ ]:
# ← your answer here: replace every ... with code
deep_quakes = quakes[...]                                  # only the rows deeper than 500 km
per_region = deep_quakes.groupby(...)["depth"].count()     # one count for each region

print(len(deep_quakes), "earthquakes deeper than 500 km")
print(per_region.sort_values(ascending=False).head(6))

In [ ]:
assert deep_quakes["depth"].min() > 500, \
    "deep_quakes should keep only the rows deeper than 500 km: > rather than >=, because two rows sit at exactly 500.0 km"
assert 500 < len(deep_quakes) < 570, \
    "deep_quakes should keep every row deeper than 500 km and nothing shallower — the threshold is 500"
assert "Indonesia" in per_region.index, "per_region should be grouped by the region column"
assert per_region.loc["Indonesia"] > 0 and per_region.sum() == len(deep_quakes), \
    "per_region should count the rows of deep_quakes, not of quakes"
print(f"✓ deep earthquakes — {len(deep_quakes)} deeper than 500 km, in {len(per_region)} regions")

Three of those names are one place: Fiji, south of the Fiji Islands and Fiji region together hold
339 of the 535, where old ocean floor has sunk hundreds of kilometres into the
mantle near Fiji. Indonesia, Argentina and the Philippines follow, far behind. Earthquakes this
deep mark only some of the places where ocean floor goes down, and almost all of them lie inside
the sinking plate.

A list of names says which places. A map says where — and because every row of the catalogue
carries its own longitude and latitude, the rows go straight onto the same map section 2 drew.

In [ ]:
plt.plot(coast.lon, coast.lat, color="grey", lw=0.5)
plt.scatter(deep_quakes["longitude"], deep_quakes["latitude"], s=8, color="firebrick")
plt.xlabel("longitude (degrees east)")
plt.ylabel("latitude (degrees north)")
plt.title(f"earthquakes deeper than 500 km ({len(deep_quakes)})")
plt.show()

Not scattered: a few short arcs. Tonga and Fiji in the southwest Pacific, cut in two by the edges
of the map because that arc straddles the date line; then Indonesia and the Philippines, Japan and
the Kuriles, and South America. Each arc is a slab of ocean floor going down.

One dot sits on its own, under southern Spain: a magnitude 6.3 at 610 km in 2010, in no arc at
all. The rule this section found holds for almost every deep earthquake, and that one is the
exception a map shows you and a table of region names does not.

## The question, answered

No: both planets have two levels, Earth's 4,500 m apart
and Mars's 5,750 m, but Earth's are its two kinds of crust, while
Mars's low half sits on crust that appears thinner, for a reason still argued about.

## Week 3 summary

### The science

**Earth's elevation has two peaks. So does Mars's. Same reason?**  
Earth's surface has two levels — ocean floor and continent, two kinds of crust floating at different heights. Mars has two as well, but not by Earth's mechanism: Mars has one plate and no ocean crust. What did make Mars's is still argued about — a giant impact, or convection in its mantle. A histogram's bin count can hide the science: too few bins merge real peaks. A longitude-latitude grid over-counts the poles, so area-weight before quoting any percentage of a planet's surface. A grid of one quantity is an array, where position is location. A list of events, each carrying different kinds of fact, is a table, where you ask by name.

- **Array.** A grid of numbers where every cell is the same kind of thing, so one line of arithmetic changes all of them at once.
- **Mask.** Comparing an array with a number asks the same question of every cell at once and hands back a grid of True and False.
- **NaN.** Where the file had nothing at all, pandas puts NaN. A NaN is a hole, not a zero.
- **Two kinds of crust.** Ocean crust is thin and dense, continental crust thick and light; both float on the mantle, and the thick, light kind floats higher.
- **Crustal dichotomy.** Mars's northern half sits kilometres lower than its southern half, and what made the step is still argued about.
- **Table.** A table with a name on every column, so you ask for data by name instead of by position.
- **Area weighting.** A longitude-latitude grid counts every square once, but a square near the pole is a sliver; weight each row by cos(latitude) before quoting a percentage.

### The code

| Function | What it does |
|---|---|
| **NumPy** | |
| `np.array(list)` | the container that does arithmetic to every number at once |
| `grid.shape` | how many rows and columns |
| `grid.size` | how many numbers altogether |
| `grid.ravel()` | lay a grid out flat, as one long line of numbers |
| `grid.sum()` | add every number up — on a mask, that counts the Trues |
| `grid.argmax()` | the position of the largest value, not the value itself |
| `np.histogram(values, bins=edges)` | the counts a histogram would draw, handed back as numbers |
| `grid[mask]` | keep only the cells where the mask is True |
| `mask & other_mask` | True only where both masks are True, the array version of and |
| **pandas** | |
| `table.info()` | every column, its type, and how many rows are not blank |
| `table.head()` | the first five rows |
| `column.isna()` | a mask marking where the file had nothing |
| `table.dropna()` | throw away every row with a hole anywhere in it |
| `column.value_counts()` | how often each value appears |
| `table.sort_values(by)` | put the rows in order by one column |
| `table.groupby(column)` | split the table into one group per value |
| `column.count() / column.median()` | how many, and the middle value |
| `table[mask]` | keep only the rows where the mask is True |
| **Matplotlib** | |
| `plt.imshow(grid, extent=[...])` | draw a whole grid as a picture, one pixel per cell |
| `plt.colorbar(label=...)` | the key that says what the colours mean |

## Homework

Three parts, all the same move as Your turn 3: build a mask with a comparison, then measure it.
Each part changes one thing — which half of the planet, where the line goes, and what is being
masked.

Class found Earth's two levels at -4375 m and 125 m, and Mars's at
-4375 m and 1375 m. That both deep levels land in the same bin says nothing
about the planets being alike, since Mars's zero is only a reference surface. A planet's **low
level** below is everything under -1000 m: the trough that class split each planet at, in Your
turn 2 and Your turn 4.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──

# Run the setup cell at the top, then re-run your own Your turn 3, where `area_share` is defined:
# it is your answer, so this cell cannot rebuild it for you. Everything else is rebuilt here.
north = latitude > 0
earth_low = earth < -1000
mars_low = mars < -1000

### ✏️ Your turn 6

**Is Earth's low level as one-sided as Mars's?**

`latitude > 0` is a mask of the northern half of a planet, so `earth_low & north` is the part of
Earth's low level that lies in the north — the `&` from section 1's mask cell.

Each share below is a division of two shares of the whole planet, and dividing one by the other
leaves a share of the low level alone: the area of the low level that is also northern, over
the area of the whole low level. Fill in the two shares.

So: is either planet's low level mostly on one side of the equator, and is Earth's as one-sided
as Mars's?

**Use these names**, because the self-check looks for them: `earth_north` and `mars_north`.

In [ ]:
# ← your answer here: replace every ... with code
north = latitude > 0                                  # every cell north of the equator
earth_low = earth < -1000                             # Earth's low level
mars_low = mars < -1000                               # Mars's low level

earth_north = area_share(..., cell_area) / area_share(..., cell_area)       # the share of Earth's low level that lies in the north
mars_north = area_share(..., cell_area) / area_share(..., cell_area)        # the same for Mars

print("Earth keeps", round(earth_north, 2), "of its low level north of the equator")
print("Mars keeps ", round(mars_north, 2), "of its low level north of the equator")

In [ ]:
assert 0.38 < earth_north < 0.41, \
    "earth_north is the AREA of Earth's low level that is also in the north, over the area of the whole low level"
assert 0.82 < mars_north < 0.85, \
    "mars_north is the same share for Mars, with Mars's own low level on both sides of the division"
print(f"✓ Homework 1 — Mars keeps {mars_north:.2f} of its low level in the north and Earth {earth_north:.2f}: "
      f"Mars's low level is one-sided, Earth's spreads over both halves")

**What the numbers mean.** Compare your printout with the explanation in the solution, published Wednesday.

### ✏️ Your turn 7

**Where does the ocean floor stop?**

Class drew two lines on Earth this week, and neither is obviously where the ocean floor stops. Sea
level is where the water happens to stand. The trough at -1000 m is where Earth has least ground,
the gap between the two humps — a fact about the distribution, not a boundary anyone drew. Pick
**one**:

- **Option A — sea level.** `line = 0`
- **Option B — the trough between Earth's two levels.** `line = -1000`

The cell below measures the share of Earth's surface below your line, and the share that is under
the sea but still above the trough — below 0 m and at or above -1000 m, both at once — which is
the ground the two options disagree about. Keep each comparison inside its own brackets,
as the cell shows, or Python cannot tell where one comparison ends and the `&` begins. Fill in
your line and the three masks.

So: how much of Earth's surface is ocean floor?

**Use these names**, because the self-check looks for them: `line`, `ocean_floor` and
`shallow_floor`.

In [ ]:
# ← your answer here: replace every ... with code
line = ...                                  # option A: 0, sea level      option B: -1000, the trough
ocean_floor = area_share(..., cell_area)               # the share of Earth's surface below your line
shallow_floor = area_share((...) & (...), cell_area)   # under the sea, but still above the trough

print("With the line at", line, "m,", round(ocean_floor, 3), "of Earth's surface is ocean floor;")
print(round(shallow_floor, 3), "of the surface is under the sea but above the trough, where A and B disagree")

In [ ]:
assert line == 0 or line == -1000, "set line to 0 (option A) or -1000 (option B)"
assert (line == 0 and 0.70 < ocean_floor < 0.72) or (line == -1000 and 0.62 < ocean_floor < 0.64), \
    "ocean_floor is the area share of the cells BELOW your line"
assert 0.081 < shallow_floor < 0.083, \
    "shallow_floor is the area share below 0 m AND at or above -1000 m, both at once"
print(f"✓ Homework 2 — with the line at {line} m, {ocean_floor:.3f} of Earth is ocean floor; "
      f"{shallow_floor:.3f} lies between the two lines")

**What the numbers mean.** Compare your printout with the explanation in the solution, published Wednesday.

### ✏️ Your turn 8

**Are the deep earthquakes spread like the shallow ones?**

Parts 1 and 2 masked a grid. This part makes the same move on the table: keep some rows with a
comparison, then measure them. The cell below keeps the earthquakes deeper than 500 km, as Your
turn 5 did, and the shallow ones — less than 70 km down, which is four out of five — then asks
what share of each set lies north of the equator. Fill in the two masks and the two shares.

So: do the deep earthquakes sit where the shallow ones do, and should these shares be
area-weighted the way Earth's low level was?

**Use these names**, because the self-check looks for them: `deep`, `shallow`, `deep_north` and
`shallow_north`.

In [ ]:
# ← your answer here: replace every ... with code
deep = quakes[...]                   # the rows deeper than 500 km, as in Your turn 5
shallow = quakes[...]                # the rows less than 70 km down
deep_north = ... / ...               # how many of the deep ones are north of the equator, over how many there are
shallow_north = ... / ...            # the same share for the shallow ones

print(f"deeper than 500 km:  {len(deep):6d} earthquakes, {deep_north:.2f} of them north of the equator")
print(f"shallower than 70 km: {len(shallow):5d} earthquakes, {shallow_north:.2f} north")

In [ ]:
assert deep["depth"].min() > 500 and shallow["depth"].max() < 70, \
    "deep keeps the rows deeper than 500 km and shallow the rows less than 70 km down"
assert 3000 < len(shallow) - len(deep) * 10 and len(shallow) > 8 * len(deep), \
    "shallow should hold the great majority of the catalogue, deep a few hundred rows"
assert 0.10 < deep_north < 0.18, \
    "deep_north is the share of DEEP that lies north of the equator — count the rows with latitude above 0 and divide by len(deep)"
assert 0.42 < shallow_north < 0.49, \
    "shallow_north is the same share for shallow, divided by len(shallow), not by len(quakes)"
print(f"✓ Homework 3 — {deep_north:.2f} of the deep earthquakes are north of the equator "
      f"against {shallow_north:.2f} of the shallow ones")

**What the numbers mean.** Compare your printout with the explanation in the solution, published Wednesday.